# CircuitOCR — Colab GPU Training & Evaluation
**PaddleOCR-VL-0.9B + Projector-Only LoRA (r=16)**

点击上方 Select Kernel → Colab → Auto Connect 连接 GPU

In [ ]:
# 1. Check GPU
!nvidia-smi
!pip install paddlepaddle-gpu -q 2>&1 | tail -1

In [ ]:
# 2. Clone repo & install deps
!git clone https://github.com/ZhangJ83/circuit-ocr-paddle.git 2>/dev/null || echo "Repo exists"
%cd circuit-ocr-paddle/circuit-ocr-dataset
!pip install paddleformers pillow opencv-python Levenshtein -q 2>&1 | tail -2

In [ ]:
# 3. Download model (skip if cached)
import os
model_path = "/content/models/PaddleOCR-VL"
if not os.path.exists(model_path):
    !mkdir -p {model_path}
    from huggingface_hub import snapshot_download
    snapshot_download("PaddlePaddle/PaddleOCR-VL", local_dir=model_path)
    print("Model downloaded")
else:
    print("Model cached")

In [ ]:
# 4. Train Projector-Only r=16 (T4 ~30min, max_dim=336)
!python scripts/train_projector_only.py 2>&1 | tail -20

In [ ]:
# 5. Eval on easy50
!cp PaddleOCR-VL-LoRA-circuit-ocr/lora_projector_only_fp16.pdparams PaddleOCR-VL-LoRA-circuit-ocr/lora_final_fp16.pdparams
!python scripts/eval_benchmark.py 
    --model_type paddleocr-vl 
    --model_name_or_path "/content/models/PaddleOCR-VL" 
    --paddle_lora_dir "PaddleOCR-VL-LoRA-circuit-ocr" 
    --data_path ocr_vl_sft-test-easy50.jsonl 
    --output_path results.jsonl 
    --max_length 30